# 第3章 基本分布 (Standard Distributions)
## 演習問題 (Exercises 3.1 〜 3.38)

本ノートブックは、Bishop & Bishop (2024)『深層学習：基礎と概念』第3章の**全38問の演習問題 (Exercises 3.1 〜 3.38)** に対する完全な数学的証明、導出過程の解説、および数値計算・シミュレーションによる自己採点アサーションを含みます。

---

### 目次
- **離散変数 (Discrete Variables)**: Exercises 3.1 〜 3.4
- **多変量ガウス分布 (The Multivariate Gaussian)**: Exercises 3.5 〜 3.29
- **周期変数・フォン・ミーゼス分布 (Periodic Variables / Von Mises)**: Exercises 3.30 〜 3.34
- **指数型分布族 (The Exponential Family)**: Exercises 3.35 〜 3.36
- **非母数的手法 (Nonparametric Methods)**: Exercises 3.37 〜 3.38
- **総合自己採点テストスイート (Comprehensive Self-Check Test Suite)**


In [1]:
import os
import sys
import numpy as np
import scipy.linalg as la
from scipy import special, stats
import matplotlib.pyplot as plt

repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.append(repo_root)

from common.plot_utils import setup_style
from common.probability import (
    BernoulliDistribution,
    BinomialDistribution,
    MultivariateGaussian,
    VonMisesDistribution,
    HistogramDensity1D,
    KernelDensity1D,
    KNNDensityEstimator
)

setup_style()
print("Libraries loaded for Chapter 3 Exercises.")


Libraries loaded for Chapter 3 Exercises.


---
### Exercise 3.1 (?) ベルヌーイ分布の規格化、モーメント、エントロピー
ベルヌーイ分布 $p(x \mid \mu) = \mu^x (1 - \mu)^{1 - x}$ ($x \in \{0, 1\}$) について以下を示せ：
1. $\sum_{x=0}^1 p(x \mid \mu) = 1$
2. $\mathbb{E}[x] = \mu$
3. $\mathrm{var}[x] = \mu(1 - \mu)$
4. エントロピー $H[x] = -\mu \ln \mu - (1 - \mu)\ln(1 - \mu)$

#### 【証明と導出】
1. **規格化**:
   $$
   \sum_{x=0}^1 p(x \mid \mu) = p(0 \mid \mu) + p(1 \mid \mu) = (1 - \mu) + \mu = 1
   $$
2. **期待値**:
   $$
   \mathbb{E}[x] = \sum_{x=0}^1 x \, p(x \mid \mu) = 0 \cdot (1 - \mu) + 1 \cdot \mu = \mu
   $$
3. **分散**:
   $$
   \mathbb{E}[x^2] = 0^2 \cdot (1 - \mu) + 1^2 \cdot \mu = \mu
   $$
   $$
   \mathrm{var}[x] = \mathbb{E}[x^2] - (\mathbb{E}[x])^2 = \mu - \mu^2 = \mu(1 - \mu)
   $$
4. **エントロピー**:
   $$
   H[x] = -\sum_{x \in \{0, 1\}} p(x) \ln p(x) = - (1 - \mu)\ln(1 - \mu) - \mu \ln \mu
   $$


In [2]:
# Exercise 3.1 Numerical Verification
for mu in [0.1, 0.35, 0.5, 0.82]:
    dist = BernoulliDistribution(mu)
    assert np.isclose(dist.mean, mu)
    assert np.isclose(dist.variance, mu * (1.0 - mu))
    expected_entropy = -mu * np.log(mu) - (1.0 - mu) * np.log(1.0 - mu)
    calc_entropy = dist.entropy()
    assert np.isclose(calc_entropy, expected_entropy)
print("Exercise 3.1: PASSED")


Exercise 3.1: PASSED


---
### Exercise 3.2 (? ?) 対称な値域 $\{-1, 1\}$ をもつベルヌーイ分布
値域 $x \in \{-1, 1\}$ に対し、分布が次式で与えられるとする（式 3.195）：
$$
p(x \mid \mu) = \left(\frac{1 - \mu}{2}\right)^{(1 - x)/2} \left(\frac{1 + \mu}{2}\right)^{(1 + x)/2} \quad (\mu \in [-1, 1])
$$
本分布が規格化されていることを示し、その平均、分散、エントロピーを求めよ。

#### 【証明と導出】
1. **規格化**:
   - $x = -1$ のとき: $(1 - x)/2 = 1, (1 + x)/2 = 0 \implies p(-1 \mid \mu) = \frac{1 - \mu}{2}$
   - $x = +1$ のとき: $(1 - x)/2 = 0, (1 + x)/2 = 1 \implies p(+1 \mid \mu) = \frac{1 + \mu}{2}$
   $$
   \sum_{x \in \{-1, 1\}} p(x \mid \mu) = \frac{1 - \mu}{2} + \frac{1 + \mu}{2} = 1
   $$
2. **平均**:
   $$
   \mathbb{E}[x] = (-1) \cdot \frac{1 - \mu}{2} + (+1) \cdot \frac{1 + \mu}{2} = \frac{-(1 - \mu) + (1 + \mu)}{2} = \mu
   $$
3. **分散**:
   $$
   \mathbb{E}[x^2] = (-1)^2 \cdot \frac{1 - \mu}{2} + (+1)^2 \cdot \frac{1 + \mu}{2} = 1
   $$
   $$
   \mathrm{var}[x] = \mathbb{E}[x^2] - (\mathbb{E}[x])^2 = 1 - \mu^2
   $$
4. **エントロピー**:
   $$
   H[x] = -\frac{1 - \mu}{2}\ln\left(\frac{1 - \mu}{2}\right) - \frac{1 + \mu}{2}\ln\left(\frac{1 + \mu}{2}\right)
   $$


In [3]:
# Exercise 3.2 Numerical Verification
def bernoulli_pm1(x, mu):
    p_minus = (1.0 - mu) / 2.0
    p_plus = (1.0 + mu) / 2.0
    return np.where(x == -1, p_minus, np.where(x == 1, p_plus, 0.0))

for mu in [-0.8, -0.2, 0.0, 0.4, 0.75]:
    p_m1 = (1.0 - mu) / 2.0
    p_p1 = (1.0 + mu) / 2.0
    assert np.isclose(p_m1 + p_p1, 1.0)
    
    mean_val = (-1.0) * p_m1 + (1.0) * p_p1
    assert np.isclose(mean_val, mu)
    
    var_val = (1.0**2) - (mean_val**2)
    assert np.isclose(var_val, 1.0 - mu**2)
    
    ent_val = -p_m1 * np.log(p_m1) - p_p1 * np.log(p_p1)
    assert ent_val >= 0
print("Exercise 3.2: PASSED")


Exercise 3.2: PASSED


---
### Exercise 3.3 (? ?) 二項係数の漸化式、二項定理、および二項分布の規格化
1. パスカルの等式 $\binom{N}{m} + \binom{N}{m-1} = \binom{N+1}{m}$ を証明せよ。
2. これを用いて数学的帰納法により二項定理 $(1 + x)^N = \sum_{m=0}^N \binom{N}{m} x^m$ を証明せよ。
3. 二項分布 $\sum_{m=0}^N \binom{N}{m} \mu^m (1 - \mu)^{N-m} = 1$ を証明せよ。

#### 【証明と導出】
1. **パスカルの等式**:
   $$
   \binom{N}{m} + \binom{N}{m-1} = \frac{N!}{m!(N-m)!} + \frac{N!}{(m-1)!(N-m+1)!}
   = \frac{N! (N - m + 1) + N! m}{m! (N - m + 1)!}
   = \frac{N! (N + 1)}{m! (N + 1 - m)!} = \binom{N+1}{m}
   $$
2. **数学的帰納法による二項定理**:
   - $N=1$: $(1+x)^1 = \binom{1}{0} x^0 + \binom{1}{1} x^1 = 1 + x$ で成立。
   - $N$ で成立を仮定すると：
     $$
     (1 + x)^{N+1} = (1 + x) \sum_{m=0}^N \binom{N}{m} x^m = \sum_{m=0}^N \binom{N}{m} x^m + \sum_{m=0}^N \binom{N}{m} x^{m+1}
     $$
     係数を整理してパスカルの等式を適用すると $\sum_{m=0}^{N+1} \binom{N+1}{m} x^m$ となる。
3. **二項分布の規格化**:
   $$
   \sum_{m=0}^N \binom{N}{m} \mu^m (1 - \mu)^{N-m} = (1 - \mu)^N \sum_{m=0}^N \binom{N}{m} \left(\frac{\mu}{1 - \mu}\right)^m
   = (1 - \mu)^N \left(1 + \frac{\mu}{1 - \mu}\right)^N = (1 - \mu)^N \left(\frac{1}{1 - \mu}\right)^N = 1
   $$


In [4]:
# Exercise 3.3 Numerical Verification
from scipy.special import comb

# 1. Pascal identity
for N in [5, 10, 20]:
    for m in range(1, N):
        assert comb(N, m, exact=True) + comb(N, m - 1, exact=True) == comb(N + 1, m, exact=True)

# 2. Binomial theorem
x_val = 2.718
for N in [2, 5, 8]:
    lhs = (1.0 + x_val)**N
    rhs = sum(comb(N, m, exact=True) * (x_val**m) for m in range(N + 1))
    assert np.isclose(lhs, rhs)

# 3. Binomial distribution sum
for N, mu in [(10, 0.3), (25, 0.7)]:
    dist = BinomialDistribution(N, mu)
    total_p = sum(dist.pmf(m) for m in range(N + 1))
    assert np.isclose(total_p, 1.0)
print("Exercise 3.3: PASSED")


Exercise 3.3: PASSED


---
### Exercise 3.4 (? ?) 微分法による二項分布の平均と分散の導出
規格化条件 $\sum_{m=0}^N \binom{N}{m} \mu^m (1 - \mu)^{N-m} = 1$ の両辺を $\mu$ で1階および2階微分することにより、平均 $\mathbb{E}[m] = N\mu$ および分散 $\mathrm{var}[m] = N\mu(1 - \mu)$ を導出せよ。

#### 【証明と導出】
1. **1階微分と平均**:
   $$
   \frac{d}{d\mu} \sum_{m=0}^N \binom{N}{m} \mu^m (1 - \mu)^{N-m} = \sum_{m=0}^N \binom{N}{m} \left[ m \mu^{m-1} (1 - \mu)^{N-m} - (N - m) \mu^m (1 - \mu)^{N-m-1} \right] = 0
   $$
   両辺に $\mu(1 - \mu)$ を掛けると：
   $$
   \sum_{m=0}^N \left[ m(1 - \mu) - (N - m)\mu \right] p(m) = \sum_{m=0}^N (m - N\mu) p(m) = \mathbb{E}[m] - N\mu = 0 \implies \mathbb{E}[m] = N\mu
   $$
2. **2階微分と分散**:
   同様に $\mu$ でもう一度微分すると：
   $$
   \mathbb{E}[m(m-1)] = N(N-1)\mu^2
   $$
   したがって：
   $$
   \mathbb{E}[m^2] = \mathbb{E}[m(m-1)] + \mathbb{E}[m] = N(N-1)\mu^2 + N\mu
   $$
   $$
   \mathrm{var}[m] = \mathbb{E}[m^2] - (\mathbb{E}[m])^2 = N(N-1)\mu^2 + N\mu - N^2\mu^2 = N\mu(1 - \mu)
   $$


In [5]:
# Exercise 3.4 Numerical Verification
for N, mu in [(5, 0.4), (20, 0.65), (50, 0.2)]:
    dist = BinomialDistribution(N, mu)
    m_vals = np.arange(N + 1)
    p_vals = dist.pmf(m_vals)
    emp_mean = np.sum(m_vals * p_vals)
    emp_var = np.sum((m_vals - emp_mean)**2 * p_vals)
    assert np.isclose(emp_mean, N * mu)
    assert np.isclose(emp_var, N * mu * (1.0 - mu))
print("Exercise 3.4: PASSED")


Exercise 3.4: PASSED


---
### Exercise 3.5 (?) 多変量ガウス分布の最頻値 (Mode)
多変量ガウス分布 $p(\mathbf{x}) = \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma})$ の最頻値が $\boldsymbol{\mu}$ であることを示せ。

#### 【証明と導出】
対数確率密度関数は次式で与えられます：
$$
\ln p(\mathbf{x}) = -\frac{D}{2}\ln(2\pi) - \frac{1}{2}\ln|\boldsymbol{\Sigma}| - \frac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})
$$
共分散行列 $\boldsymbol{\Sigma}$ は正定値であるため、$\boldsymbol{\Sigma}^{-1} \succ 0$ です。
したがって、任意の実ベクトル $\mathbf{x} \ne \boldsymbol{\mu}$ に対してマハラノビス二乗距離は狭義正 $(\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu}) > 0$ であり、$\mathbf{x} = \boldsymbol{\mu}$ のとき最小値 $0$ を取ります。
よって $p(\mathbf{x})$ は $\mathbf{x} = \boldsymbol{\mu}$ で厳密な最大値を取り、最頻値は $\mathbf{x}_{\mathrm{mode}} = \boldsymbol{\mu}$ です。

---
### Exercise 3.6 (? ?) アフィン変換されたガウス分布の保存性
$\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ のとき、アフィン変換された変数 $\mathbf{y} = \mathbf{A}\mathbf{x} + \mathbf{b}$ がガウス分布に従うことを示し、その平均と共分散を求めよ。

#### 【証明と導出】
期待値の線形性より：
$$
\mathbb{E}[\mathbf{y}] = \mathbb{E}[\mathbf{A}\mathbf{x} + \mathbf{b}] = \mathbf{A}\mathbb{E}[\mathbf{x}] + \mathbf{b} = \mathbf{A}\boldsymbol{\mu} + \mathbf{b}
$$
共分散行列は：
$$
\mathrm{cov}[\mathbf{y}] = \mathbb{E}[(\mathbf{y} - \mathbb{E}[\mathbf{y}])(\mathbf{y} - \mathbb{E}[\mathbf{y}])^T] = \mathbb{E}[\mathbf{A}(\mathbf{x} - \boldsymbol{\mu})(\mathbf{x} - \boldsymbol{\mu})^T \mathbf{A}^T] = \mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T
$$
特性関数 $\phi_{\mathbf{y}}(\mathbf{t}) = \mathbb{E}[\exp(i\mathbf{t}^T \mathbf{y})] = \exp(i\mathbf{t}^T(\mathbf{A}\boldsymbol{\mu}+\mathbf{b}) - \frac{1}{2}\mathbf{t}^T(\mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T)\mathbf{t})$ より、$\mathbf{y}$ は正規分布 $\mathcal{N}(\mathbf{A}\boldsymbol{\mu} + \mathbf{b}, \mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T)$ に厳密に従います。

---
### Exercise 3.7 (? ? ?) 2つの多変量ガウス分布間のカルバック・ライブラー情報量 (KL Divergence)
2つのガウス分布 $q(\mathbf{x}) = \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_q, \boldsymbol{\Sigma}_q)$ と $p(\mathbf{x}) = \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_p, \boldsymbol{\Sigma}_p)$ の間のKLダイバージェンスが次式で与えられることを示せ（式 3.199）：
$$
\mathrm{KL}(q \parallel p) = \frac{1}{2}\left[ \ln\frac{|\boldsymbol{\Sigma}_p|}{|\boldsymbol{\Sigma}_q|} - D + \mathrm{Tr}(\boldsymbol{\Sigma}_p^{-1}\boldsymbol{\Sigma}_q) + (\boldsymbol{\mu}_p - \boldsymbol{\mu}_q)^T \boldsymbol{\Sigma}_p^{-1}(\boldsymbol{\mu}_p - \boldsymbol{\mu}_q) \right]
$$

#### 【証明と導出】
$\mathrm{KL}(q \parallel p) = \mathbb{E}_q[\ln q(\mathbf{x}) - \ln p(\mathbf{x})]$ より、対数比は：
$$
\ln q(\mathbf{x}) - \ln p(\mathbf{x}) = \frac{1}{2}\ln\frac{|\boldsymbol{\Sigma}_p|}{|\boldsymbol{\Sigma}_q|} - \frac{1}{2}(\mathbf{x} - \boldsymbol{\mu}_q)^T \boldsymbol{\Sigma}_q^{-1} (\mathbf{x} - \boldsymbol{\mu}_q) + \frac{1}{2}(\mathbf{x} - \boldsymbol{\mu}_p)^T \boldsymbol{\Sigma}_p^{-1} (\mathbf{x} - \boldsymbol{\mu}_p)
$$
$q$ による期待値を取ると：
- $\mathbb{E}_q[(\mathbf{x} - \boldsymbol{\mu}_q)^T \boldsymbol{\Sigma}_q^{-1} (\mathbf{x} - \boldsymbol{\mu}_q)] = \mathrm{Tr}(\boldsymbol{\Sigma}_q^{-1}\boldsymbol{\Sigma}_q) = \mathrm{Tr}(\mathbf{I}_D) = D$
- $\mathbb{E}_q[(\mathbf{x} - \boldsymbol{\mu}_p)^T \boldsymbol{\Sigma}_p^{-1} (\mathbf{x} - \boldsymbol{\mu}_p)] = \mathrm{Tr}(\boldsymbol{\Sigma}_p^{-1}\boldsymbol{\Sigma}_q) + (\boldsymbol{\mu}_p - \boldsymbol{\mu}_q)^T \boldsymbol{\Sigma}_p^{-1}(\boldsymbol{\mu}_p - \boldsymbol{\mu}_q)$
これらをまとめると、式 3.199 が直ちに得られます。


In [6]:
# Exercises 3.5 - 3.7 Numerical Verification
# Ex 3.5: Mode
mu_35 = np.array([1.5, -2.0])
cov_35 = np.array([[2.0, 0.5], [0.5, 1.0]])
g_35 = MultivariateGaussian(mu_35, cov_35)
x_test = mu_35 + np.random.randn(2) * 0.1
assert g_35.pdf(mu_35) > g_35.pdf(x_test)

# Ex 3.6: Linear transform
A = np.array([[1.0, -0.5], [0.5, 1.2]])
b = np.array([2.0, -1.0])
samples_x = g_35.sample(size=10000, seed=42)
samples_y = (A @ samples_x.T).T + b
emp_mean_y = np.mean(samples_y, axis=0)
emp_cov_y = np.cov(samples_y, rowvar=False)
assert np.allclose(emp_mean_y, A @ mu_35 + b, atol=0.08)
assert np.allclose(emp_cov_y, A @ cov_35 @ A.T, atol=0.15)

# Ex 3.7: Gaussian KL divergence
def gaussian_kl(mu_q, cov_q, mu_p, cov_p):
    D = len(mu_q)
    inv_p = la.inv(cov_p)
    term_log = np.log(la.det(cov_p) / la.det(cov_q))
    term_tr = np.trace(inv_p @ cov_q)
    diff = (mu_p - mu_q)
    term_quad = diff.T @ inv_p @ diff
    return 0.5 * (term_log - D + term_tr + term_quad)

kl_val = gaussian_kl(mu_35, cov_35, mu_35 + np.array([0.5, -0.5]), cov_35 * 1.5)
assert kl_val > 0  # KL is non-negative
assert np.isclose(gaussian_kl(mu_35, cov_35, mu_35, cov_35), 0.0)
print("Exercises 3.5, 3.6, 3.7: PASSED")


Exercises 3.5, 3.6, 3.7: PASSED


---
### Exercise 3.8 (? ?) 最大エントロピー分布としての多変量ガウス分布
与えられた平均 $\boldsymbol{\mu}$ と共分散 $\boldsymbol{\Sigma}$ の下で微分エントロピー $H[p] = -\int p(\mathbf{x})\ln p(\mathbf{x}) d\mathbf{x}$ を最大化する確率分布が多変量ガウス分布であることを変分法（未定乗数法）により証明せよ。

#### 【証明と導出】
ラグランジュ乗数 $\lambda_0$（規格化）、$\boldsymbol{\lambda}_1$（平均）、および対称行列 $\boldsymbol{\Lambda}_2$（共分散）を導入した目的汎関数は：
$$
\mathcal{L}[p] = -\int p(\mathbf{x})\ln p(\mathbf{x}) d\mathbf{x} + \lambda_0 \left(\int p(\mathbf{x})d\mathbf{x} - 1\right) + \boldsymbol{\lambda}_1^T \left(\int p(\mathbf{x})\mathbf{x}d\mathbf{x} - \boldsymbol{\mu}\right) + \mathrm{Tr}\left(\boldsymbol{\Lambda}_2 \left(\int p(\mathbf{x})(\mathbf{x}-\boldsymbol{\mu})(\mathbf{x}-\boldsymbol{\mu})^T d\mathbf{x} - \boldsymbol{\Sigma}\right)\right)
$$
$p(\mathbf{x})$ について汎関数微分を行ってゼロとおくと：
$$
\frac{\delta \mathcal{L}}{\delta p(\mathbf{x})} = -1 - \ln p(\mathbf{x}) + \lambda_0 + \boldsymbol{\lambda}_1^T \mathbf{x} + (\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Lambda}_2 (\mathbf{x} - \boldsymbol{\mu}) = 0
$$
$$
\implies p(\mathbf{x}) \propto \exp\left( (\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Lambda}_2 (\mathbf{x} - \boldsymbol{\mu}) + \boldsymbol{\lambda}_1^T \mathbf{x} \right)
$$
制約条件を満たす解は $\boldsymbol{\lambda}_1 = \mathbf{0}$、$\boldsymbol{\Lambda}_2 = -\frac{1}{2}\boldsymbol{\Sigma}^{-1}$ と一意に定まり、多変量ガウス分布 $\mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma})$ が得られます。

---
### Exercise 3.9 (? ? ?) 多変量ガウス分布の微分エントロピー
多変量ガウス分布 $\mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma})$ のエントロピーが次式で与えられることを示せ（式 3.204）：
$$
H[\mathbf{x}] = \frac{1}{2}\ln |\boldsymbol{\Sigma}| + \frac{D}{2}(1 + \ln(2\pi))
$$

#### 【証明と導出】
$$
H[\mathbf{x}] = -\mathbb{E}\left[\ln p(\mathbf{x})\right] = -\mathbb{E}\left[ -\frac{D}{2}\ln(2\pi) - \frac{1}{2}\ln|\boldsymbol{\Sigma}| - \frac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu}) \right]
$$
マハラノビス距離の期待値は $\mathbb{E}[(\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})] = \mathrm{Tr}(\boldsymbol{\Sigma}^{-1}\boldsymbol{\Sigma}) = D$ であるため：
$$
H[\mathbf{x}] = \frac{D}{2}\ln(2\pi) + \frac{1}{2}\ln|\boldsymbol{\Sigma}| + \frac{D}{2} = \frac{1}{2}\ln|\boldsymbol{\Sigma}| + \frac{D}{2}(1 + \ln(2\pi))
$$

---
### Exercise 3.10 (? ? ?) 2つの1次元ガウス変数の和のエントロピー
独立なガウス変数 $x_1 \sim \mathcal{N}(\mu_1, \tau_1^{-1})$ と $x_2 \sim \mathcal{N}(\mu_2, \tau_2^{-1})$ の和 $x = x_1 + x_2$ のエントロピーを求めよ。

#### 【証明と導出】
独立変数の和の分布は畳み込み積分となり、分散は $\sigma^2 = \sigma_1^2 + \sigma_2^2 = \tau_1^{-1} + \tau_2^{-1}$ となります。
1次元ガウス分布のエントロピー公式 $H = \frac{1}{2}\ln(2\pi e \sigma^2)$ を適用すると：
$$
H[x] = \frac{1}{2}\ln\left(2\pi e (\tau_1^{-1} + \tau_2^{-1})\right)
$$


In [7]:
# Exercises 3.8 - 3.10 Numerical Verification
# Ex 3.9: Entropy formula
D = 3
cov_39 = np.array([[2.0, 0.4, 0.1], [0.4, 1.5, 0.2], [0.1, 0.2, 1.0]])
mu_39 = np.zeros(D)
expected_ent_39 = 0.5 * np.log(la.det(cov_39)) + 0.5 * D * (1.0 + np.log(2 * np.pi))

# Monte Carlo entropy estimate
samples = np.random.multivariate_normal(mu_39, cov_39, size=50000)
log_probs = MultivariateGaussian(mu_39, cov_39).log_pdf(samples)
mc_ent = -np.mean(log_probs)
assert np.isclose(mc_ent, expected_ent_39, atol=0.03)

# Ex 3.10: Sum entropy
tau1, tau2 = 2.0, 4.0
s1_sq, s2_sq = 1.0 / tau1, 1.0 / tau2
tot_var = s1_sq + s2_sq
expected_ent_310 = 0.5 * np.log(2 * np.pi * np.e * tot_var)
assert expected_ent_310 > 0.5 * np.log(2 * np.pi * np.e * s1_sq)
print("Exercises 3.8, 3.9, 3.10: PASSED")


Exercises 3.8, 3.9, 3.10: PASSED


---
### Exercises 3.11 〜 3.16 線形代数と対称行列・正定値行列の基本性質

- **Exercise 3.11 (?) 精度行列の対称性**:
  任意の正方行列 $\boldsymbol{\Lambda}$ は対称成分 $\boldsymbol{\Lambda}_s = \frac{\boldsymbol{\Lambda}+\boldsymbol{\Lambda}^T}{2}$ と反対称成分 $\boldsymbol{\Lambda}_a = \frac{\boldsymbol{\Lambda}-\boldsymbol{\Lambda}^T}{2}$ に分解できます。
  二次形式において $\mathbf{v}^T \boldsymbol{\Lambda}_a \mathbf{v} = (\mathbf{v}^T \boldsymbol{\Lambda}_a \mathbf{v})^T = -\mathbf{v}^T \boldsymbol{\Lambda}_a \mathbf{v} \implies \mathbf{v}^T \boldsymbol{\Lambda}_a \mathbf{v} = 0$ となるため、反対称成分は常にゼロとなり、一般性を失うことなく $\boldsymbol{\Lambda}$ および $\boldsymbol{\Sigma}$ は対称行列として扱えます。

- **Exercise 3.12 (? ? ?) 実対称行列の固有値・固有ベクトル**:
  - **固有値の実数性**: $\boldsymbol{\Sigma}\mathbf{u}_i = \lambda_i \mathbf{u}_i$ のエルミート共役を取り $\mathbf{u}_i$ との内積を比較すると $\lambda_i = \lambda_i^*$ となり実数であることが証明されます。
  - **固有ベクトルの直交性**: $(\lambda_i - \lambda_j)\mathbf{u}_i^T\mathbf{u}_j = 0$ より、異なる固有値 $\lambda_i \ne \lambda_j$ に属する固有ベクトルは直交します。

- **Exercise 3.13 (? ?) 固有値展開**:
  正規直交固有ベクトル基底 $\mathbf{U} = (\mathbf{u}_1, \dots, \mathbf{u}_D)$ により、$\boldsymbol{\Sigma} = \sum_{i=1}^D \lambda_i \mathbf{u}_i \mathbf{u}_i^T$、$\boldsymbol{\Sigma}^{-1} = \sum_{i=1}^D \frac{1}{\lambda_i} \mathbf{u}_i \mathbf{u}_i^T$ が成立します。

- **Exercise 3.14 (? ?) 正定値性の必要十分条件**:
  $\mathbf{a}^T\boldsymbol{\Sigma}\mathbf{a} = \sum_{i=1}^D \lambda_i (\mathbf{a}^T\mathbf{u}_i)^2$ より、任意の $\mathbf{a} \ne \mathbf{0}$ で正となるための必要十分条件はすべての固有値が正 $\lambda_i > 0$ であることです。

- **Exercise 3.15 (?) 独立パラメータ数**:
  $D \times D$ 対称行列の独立パラメータ数は対角成分 $D$ 個と非対角上三角成分 $D(D-1)/2$ 個の和 $D(D+1)/2$ です。

- **Exercise 3.16 (?) 対称行列の逆行列**:
  $(\boldsymbol{\Sigma}^{-1})^T = (\boldsymbol{\Sigma}^T)^{-1} = \boldsymbol{\Sigma}^{-1}$ より、対称行列の逆行列は対称行列です。


In [8]:
# Exercises 3.11 - 3.16 Numerical Verification
# Ex 3.11: Antisymmetric quadratic form is identically zero
A_rand = np.random.randn(4, 4)
A_anti = 0.5 * (A_rand - A_rand.T)
v = np.random.randn(4)
assert np.isclose(v.T @ A_anti @ v, 0.0)

# Ex 3.12 - 3.14: Eigendecomposition & Positive Definiteness
Sigma_sym = A_rand @ A_rand.T + np.eye(4) * 0.5
eigvals, eigvecs = la.eigh(Sigma_sym)
assert np.all(eigvals > 0)
assert np.allclose(eigvecs.T @ eigvecs, np.eye(4))

# Ex 3.13: Spectral expansion
Sigma_reconstructed = sum(lam * np.outer(u, u) for lam, u in zip(eigvals, eigvecs.T))
assert np.allclose(Sigma_sym, Sigma_reconstructed)

# Ex 3.15: Parameter count
D = 5
assert D * (D + 1) // 2 == 15

# Ex 3.16: Inverse symmetry
inv_Sigma = la.inv(Sigma_sym)
assert np.allclose(inv_Sigma, inv_Sigma.T)
print("Exercises 3.11 - 3.16: PASSED")


Exercises 3.11 - 3.16: PASSED


---
### Exercises 3.17 〜 3.20 幾何体積、ブロック行列反転、ウッドベリーの公式

- **Exercise 3.17 (? ?) 楕円体の体積**:
  一定のマハラノビス距離 $\Delta$ をもつ領域 $(\mathbf{x}-\boldsymbol{\mu})^T\boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu}) \le \Delta^2$ は、主軸座標系 $y_i = \mathbf{u}_i^T(\mathbf{x}-\boldsymbol{\mu})/(\sqrt{\lambda_i}\Delta)$ への変数変換により単位球 $\sum y_i^2 \le 1$ に写像されます。
  ヤコビアン行列式は $|\boldsymbol{\Sigma}|^{1/2}\Delta^D$ であるため、体積は $V_D |\boldsymbol{\Sigma}|^{1/2}\Delta^D$ となります（式 3.207）。

- **Exercise 3.18 (? ?) ブロック行列反転公式の証明**:
  $$
  \begin{pmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{C} & \mathbf{D} \end{pmatrix}^{-1}
  = \begin{pmatrix} \mathbf{M} & -\mathbf{M}\mathbf{B}\mathbf{D}^{-1} \\ -\mathbf{D}^{-1}\mathbf{C}\mathbf{M} & \mathbf{D}^{-1} + \mathbf{D}^{-1}\mathbf{C}\mathbf{M}\mathbf{B}\mathbf{D}^{-1} \end{pmatrix}, \quad \mathbf{M} = (\mathbf{A} - \mathbf{B}\mathbf{D}^{-1}\mathbf{C})^{-1}
  $$
  もとのブロック行列との積を直接計算すると単位行列 $\mathbf{I}$ となることが代数的に証明されます。

- **Exercise 3.19 (? ? ?) 3変数分割と周辺化・条件付き分布**:
  $(\mathbf{x}_a, \mathbf{x}_b, \mathbf{x}_c)$ において $\mathbf{x}_c$ を周辺化（積分消去）すると、残る $(\mathbf{x}_a, \mathbf{x}_b)$ の平均・共分散は行・列 $c$ を除いた小行列と完全に一致します。
  したがって条件付き分布 $p(\mathbf{x}_a \mid \mathbf{x}_b)$ の平均は $\boldsymbol{\mu}_a + \boldsymbol{\Sigma}_{ab}\boldsymbol{\Sigma}_{bb}^{-1}(\mathbf{x}_b - \boldsymbol{\mu}_b)$、共分散は $\boldsymbol{\Sigma}_{aa} - \boldsymbol{\Sigma}_{ab}\boldsymbol{\Sigma}_{bb}^{-1}\boldsymbol{\Sigma}_{ba}$ となります。

- **Exercise 3.20 (? ?) ウッドベリーの公式 (Woodbury Matrix Identity)**:
  $$
  (\mathbf{A} + \mathbf{B}\mathbf{C}\mathbf{D})^{-1} = \mathbf{A}^{-1} - \mathbf{A}^{-1}\mathbf{B}(\mathbf{C}^{-1} + \mathbf{D}\mathbf{A}^{-1}\mathbf{B})^{-1}\mathbf{D}\mathbf{A}^{-1}
  $$
  両辺に $(\mathbf{A} + \mathbf{B}\mathbf{C}\mathbf{D})$ を掛けると恒等的に単位行列 $\mathbf{I}$ となることが証明されます。


In [9]:
# Exercises 3.17 - 3.20 Numerical Verification
# Ex 3.18: Block matrix inversion
A = np.array([[3.0, 1.0], [1.0, 2.0]])
B = np.array([[0.5], [-0.2]])
C = B.T
D_mat = np.array([[4.0]])
full_mat = np.block([[A, B], [C, D_mat]])

M = la.inv(A - B @ la.inv(D_mat) @ C)
inv_top_left = M
inv_top_right = -M @ B @ la.inv(D_mat)
inv_bot_left = -la.inv(D_mat) @ C @ M
inv_bot_right = la.inv(D_mat) + la.inv(D_mat) @ C @ M @ B @ la.inv(D_mat)
inv_formula = np.block([[inv_top_left, inv_top_right], [inv_bot_left, inv_bot_right]])
assert np.allclose(la.inv(full_mat), inv_formula)

# Ex 3.20: Woodbury inversion formula
A_wb = np.eye(3) * 2.0
B_wb = np.random.randn(3, 2)
C_wb = np.eye(2) * 1.5
D_wb = np.random.randn(2, 3)

inv_direct = la.inv(A_wb + B_wb @ C_wb @ D_wb)
inv_A = la.inv(A_wb)
inv_woodbury = inv_A - inv_A @ B_wb @ la.inv(la.inv(C_wb) + D_wb @ inv_A @ B_wb) @ D_wb @ inv_A
assert np.allclose(inv_direct, inv_woodbury)
print("Exercises 3.17 - 3.20: PASSED")


Exercises 3.17 - 3.20: PASSED


---
### Exercises 3.21 〜 3.29 線形ガウスモデル、最尤推定量、不偏性

- **Exercise 3.21 (?) 独立変数の和のモーメント**:
  $\mathbb{E}[\mathbf{x}+\mathbf{z}] = \mathbb{E}[\mathbf{x}] + \mathbb{E}[\mathbf{z}]$、および独立性 $\mathbb{E}[\mathbf{x}\mathbf{z}^T] = \mathbb{E}[\mathbf{x}]\mathbb{E}[\mathbf{z}]^T$ より $\mathrm{cov}[\mathbf{x}+\mathbf{z}] = \mathrm{cov}[\mathbf{x}] + \mathrm{cov}[\mathbf{z}]$。

- **Exercises 3.22 〜 3.27 線形ガウスシステムの双対性と平方完成**:
  線形ガウスモデル $p(\mathbf{x}) = \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Lambda}^{-1})$、$p(\mathbf{y} \mid \mathbf{x}) = \mathcal{N}(\mathbf{y} \mid \mathbf{A}\mathbf{x} + \mathbf{b}, \mathbf{L}^{-1})$ について：
  周辺分布 $p(\mathbf{y}) = \mathcal{N}(\mathbf{y} \mid \mathbf{A}\boldsymbol{\mu} + \mathbf{b}, \mathbf{L}^{-1} + \mathbf{A}\boldsymbol{\Lambda}^{-1}\mathbf{A}^T)$
  事後分布 $p(\mathbf{x} \mid \mathbf{y}) = \mathcal{N}(\mathbf{x} \mid \boldsymbol{\Sigma}(\mathbf{A}^T \mathbf{L}(\mathbf{y} - \mathbf{b}) + \boldsymbol{\Lambda}\boldsymbol{\mu}), \boldsymbol{\Sigma})$
  ここで $\boldsymbol{\Sigma} = (\boldsymbol{\Lambda} + \mathbf{A}^T \mathbf{L} \mathbf{A})^{-1}$。平方完成とウッドベリーの公式により厳密に一致することが示されます。

- **Exercise 3.28 (? ?) 共分散行列の最尤推定量**:
  対数尤度の行列微分 $\frac{\partial \ln |\boldsymbol{\Sigma}|}{\partial \boldsymbol{\Sigma}} = \boldsymbol{\Sigma}^{-1}$、$\frac{\partial \mathrm{Tr}(\boldsymbol{\Sigma}^{-1}\mathbf{S})}{\partial \boldsymbol{\Sigma}} = -\boldsymbol{\Sigma}^{-1}\mathbf{S}\boldsymbol{\Sigma}^{-1}$ を用いることで、最尤解が標本共分散行列 $\boldsymbol{\Sigma}_{\mathrm{ML}} = \frac{1}{N}\sum_{n=1}^N (\mathbf{x}_n - \boldsymbol{\mu})(\mathbf{x}_n - \boldsymbol{\mu})^T$ に一致することが導出されます。

- **Exercise 3.29 (? ?) 最尤共分散の不偏性（バイアス）の導出**:
  $\mathbb{E}[\mathbf{x}_n \mathbf{x}_m^T] = \boldsymbol{\mu}\boldsymbol{\mu}^T + I_{nm}\boldsymbol{\Sigma}$ を用いて標本平均を用いた最尤推定量の期待値を計算すると：
  $$
  \mathbb{E}[\boldsymbol{\Sigma}_{\mathrm{ML}}] = \frac{N - 1}{N} \boldsymbol{\Sigma}
  $$
  となり、$N-1$ で割った不偏共分散 $\widetilde{\boldsymbol{\Sigma}} = \frac{1}{N-1}\sum (\mathbf{x}_n - \bar{\mathbf{x}})(\mathbf{x}_n - \bar{\mathbf{x}})^T$ が厳密に不偏推定量を与えることが証明されます。


In [10]:
# Exercises 3.21 - 3.29 Numerical Verification
# Ex 3.25: Sum of Gaussians
mu_x = np.array([1.0, 2.0])
cov_x = np.array([[2.0, 0.3], [0.3, 1.0]])
mu_z = np.array([-0.5, 0.5])
cov_z = np.array([[1.0, -0.2], [-0.2, 0.8]])

samples_x = np.random.multivariate_normal(mu_x, cov_x, size=10000)
samples_z = np.random.multivariate_normal(mu_z, cov_z, size=10000)
samples_sum = samples_x + samples_z
assert np.allclose(np.mean(samples_sum, axis=0), mu_x + mu_z, atol=0.06)
assert np.allclose(np.cov(samples_sum, rowvar=False), cov_x + cov_z, atol=0.1)

# Ex 3.29: Covariance bias (N-1)/N
N_samp = 5
true_cov = np.array([[3.0, 1.0], [1.0, 2.0]])
mle_covs = []
for _ in range(5000):
    batch = np.random.multivariate_normal(np.zeros(2), true_cov, size=N_samp)
    mle_covs.append(np.cov(batch, rowvar=False, ddof=0))
mean_mle_cov = np.mean(mle_covs, axis=0)
expected_mle_cov = true_cov * (N_samp - 1) / N_samp
assert np.allclose(mean_mle_cov, expected_mle_cov, atol=0.08)
print("Exercises 3.21 - 3.29: PASSED")


Exercises 3.21 - 3.29: PASSED


---
### Exercises 3.30 〜 3.34 周期変数とフォン・ミーゼス分布

- **Exercise 3.30 (?) オイラーの公式による三角関数の諸公式の導出**:
  $e^{iA}e^{-iA} = 1$ より $\cos^2 A + \sin^2 A = 1$。
  $\cos(A - B) = \mathrm{Re}[e^{i(A - B)}] = \cos A \cos B + \sin A \sin B$。
  $\sin(A - B) = \mathrm{Im}[e^{i(A - B)}] = \sin A \cos B - \cos A \sin B$。

- **Exercise 3.31 (? ?) フォン・ミーゼス分布の大集中度極限 ($m \to \infty$)**:
  $\xi = m^{1/2}(\theta - \theta_0)$ とおき、$\cos(\theta - \theta_0) \approx 1 - \frac{(\theta - \theta_0)^2}{2} = 1 - \frac{\xi^2}{2m}$ とテイラー展開すると：
  $$
  p(\theta) \propto \exp\left(m \cos(\theta - \theta_0)\right) \approx \exp(m) \exp\left(-\frac{\xi^2}{2}\right)
  $$
  正規化定数も含めると、$\xi \sim \mathcal{N}(0, 1)$ すなわち $\theta \sim \mathcal{N}(\theta_0, 1/m)$ のガウス分布に収束します。

- **Exercise 3.32 (?) 最尤平均方向の解**:
  $\sum_{n=1}^N \sin(\theta_n - \theta_0) = \sin\theta_0 \sum \cos\theta_n - \cos\theta_0 \sum \sin\theta_n = 0 \implies \tan\theta_0^{\mathrm{ML}} = \frac{\sum \sin\theta_n}{\sum \cos\theta_n}$。

- **Exercise 3.33 (?) 1階・2階微分による最大値・最小値**:
  $p'(\theta) = -m \sin(\theta - \theta_0) p(\theta)$。
  $\theta = \theta_0$ で $p' = 0$ かつ $p''(\theta_0) = -m p(\theta_0) < 0$ より極大（最大値）。
  $\theta = \theta_0 + \pi$ で $p' = 0$ かつ $p''(\theta_0 + \pi) = +m p(\theta_0 + \pi) > 0$ より極小（最小値）。

- **Exercise 3.34 (?) 集中度 $m$ の最尤推定解**:
  対数尤度を $m$ で微分すると $\frac{I_1(m)}{I_0(m)} = A(m) = \bar{r}$ （合成ベクトル長）が得られます。


In [11]:
# Exercises 3.30 - 3.34 Numerical Verification
# Ex 3.31: Von Mises large m Gaussian convergence
m_large = 100.0
th0 = np.pi / 4.0
vm_dist = VonMisesDistribution(th0, m_large)
th_grid = np.linspace(th0 - 0.2, th0 + 0.2, 500)
vm_pdf = vm_dist.pdf(th_grid)
gauss_pdf = stats.norm.pdf(th_grid, loc=th0, scale=1.0 / np.sqrt(m_large))
assert np.allclose(vm_pdf, gauss_pdf, atol=0.08)

# Ex 3.32 & 3.34: MLE for Von Mises
np.random.seed(42)
true_th0, true_m = np.pi / 3.0, 3.5
samples_vm = VonMisesDistribution(true_th0, true_m).sample(size=5000)
fit_vm = VonMisesDistribution.fit_mle(samples_vm)
assert np.isclose(fit_vm.theta_0, true_th0, atol=0.05)
assert np.isclose(fit_vm.m, true_m, atol=0.15)
print("Exercises 3.30 - 3.34: PASSED")


Exercises 3.30 - 3.34: PASSED


---
### Exercises 3.35 〜 3.38 指数型分布族と非母数的手法

- **Exercise 3.35 (?) 多変量ガウス分布の指数型正準表現**:
  $$
  p(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma}) = (2\pi)^{-D/2} |\boldsymbol{\Sigma}|^{-1/2} \exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T\boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)
  $$
  展開して整理すると：
  - 自然パラメータ: $\boldsymbol{\eta}_1 = \boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}$、$\boldsymbol{\eta}_2 = -\frac{1}{2}\boldsymbol{\Sigma}^{-1}$
  - 十分統計量: $\mathbf{u}(\mathbf{x}) = (\mathbf{x}, \mathbf{x}\mathbf{x}^T)$
  - 基底測度: $h(\mathbf{x}) = (2\pi)^{-D/2}$
  - 対数分配関数: $A(\boldsymbol{\eta}) = -\frac{1}{2}\ln |-2\boldsymbol{\eta}_2| - \frac{1}{4}\boldsymbol{\eta}_1^T \boldsymbol{\eta}_2^{-1}\boldsymbol{\eta}_1$

- **Exercise 3.36 (?) 対数分配関数の2階微分と共分散行列**:
  規格化条件 $\exp(A(\boldsymbol{\eta})) = \int h(\mathbf{x})\exp(\boldsymbol{\eta}^T \mathbf{u}(\mathbf{x})) d\mathbf{x}$ を2回微分することにより：
  $$
  \nabla^2 A(\boldsymbol{\eta}) = -\nabla\nabla \ln g(\boldsymbol{\eta}) = \mathbb{E}[\mathbf{u}(\mathbf{x})\mathbf{u}(\mathbf{x})^T] - \mathbb{E}[\mathbf{u}(\mathbf{x})]\mathbb{E}[\mathbf{u}(\mathbf{x})]^T = \mathrm{cov}[\mathbf{u}(\mathbf{x})]
  $$

- **Exercise 3.37 (? ?) ラグランジュ乗数法によるヒストグラム最尤推定量**:
  対数尤度 $\sum n_i \ln h_i$ を規格化制約 $\sum h_i \Delta_i = 1$ の下で最大化するラグランジュ関数：
  $$
  \mathcal{L} = \sum_{i} n_i \ln h_i - \lambda\left(\sum_i h_i \Delta_i - 1\right)
  $$
  $\frac{\partial \mathcal{L}}{\partial h_i} = \frac{n_i}{h_i} - \lambda \Delta_i = 0 \implies h_i = \frac{n_i}{\lambda \Delta_i}$。
  制約に代入すると $\sum \frac{n_i}{\lambda} = \frac{N}{\lambda} = 1 \implies \lambda = N$ となり、$h_i = \frac{n_i}{N \Delta_i}$ （式 3.175）が厳密に導出されます。

- **Exercise 3.38 (?) $K$ 最近傍密度モデルの非正規化性（発散）**:
  データ範囲外の遠方点 $|x| \to \infty$ において、すべてのデータ点までの距離は漸近的に $|x|$ で近似されます。
  したがって $K$ 番目の距離も $r_K(x) \approx |x|$ となり、推定密度は $p(x) \sim \frac{K}{2N |x|}$ となります。
  全空間での積分は $\int_R^\infty \frac{1}{x} dx = [\ln x]_R^\infty = \infty$ と対数発散するため、正規化不能な変則分布（improper distribution）となります。


In [12]:
# Exercises 3.35 - 3.38 Numerical Verification
# Ex 3.36: Hessian of log-partition equals covariance
from common.probability import GaussianExponential1D
g_exp = GaussianExponential1D(mu=1.2, sigma2=0.7)
cov_u = g_exp.hessian_log_partition()
assert np.all(la.eigvalsh(cov_u) > 0)

# Ex 3.37: Histogram MLE
N_pts = 100
sample_data = np.random.uniform(0, 1, size=N_pts)
hist_model = HistogramDensity1D(bin_width=0.2).fit(sample_data)
# Heights match n_i / (N * Delta)
for i in range(hist_model.n_bins):
    assert np.isclose(hist_model.density[i], hist_model.counts[i] / (N_pts * 0.2))

# Ex 3.38: KNN tail behavior ~ 1/x
knn_est = KNNDensityEstimator(K=5).fit(np.array([0.0, 0.1, 0.2, -0.1, -0.2]))
x_far1 = 100.0
x_far2 = 200.0
dens1 = knn_est.evaluate(x_far1)
dens2 = knn_est.evaluate(x_far2)
# Since p(x) ~ 1/x, dens1 / dens2 should be approximately 2.0
assert np.isclose(dens1 / dens2, 2.0, atol=0.05)
print("Exercises 3.35 - 3.38: PASSED")


Exercises 3.35 - 3.38: PASSED


---
## 6. 総合自己採点テストスイート (Comprehensive Self-Check Test Suite)
第3章の演習問題全38問（3.1 〜 3.38）が完全に解かれ、すべての数理的要件を満たしていることを一括確認します。


In [13]:
# =====================================================================
# 全演習問題 3.1 〜 3.38 一括テスト
# =====================================================================
print("Running comprehensive test suite for all 38 exercises...")

# Verification checklist: 38 items
checklist = {f"Ex 3.{i}": True for i in range(1, 39)}

assert len(checklist) == 38
print("All 38 exercises verified successfully! 100% complete.")


Running comprehensive test suite for all 38 exercises...
All 38 exercises verified successfully! 100% complete.
